In [7]:
!pip install -U python-jobspy
!pip install beautifulsoup4 
!pip install requests
!pip install pandas
!pip install numpy
!pip install openpyxl
!pip install xlrd
!pip install xlwt
!pip install xlutils

In [ ]:
import sqlite3
from datetime import datetime, timedelta
from jobspy import scrape_jobs
import pandas as pd
import time
import logging
from typing import List, Dict, Any
import random

class JobScraperETL:
    def __init__(self, db_path: str = "jobs.db"):
        """Initialize the ETL pipeline with database connection."""
        self.db_path = db_path
        self.setup_logging()
        self.setup_database()
        
        # List of proxy servers - replace with your actual proxies
        self.proxies = [
            "localhost"  # Add your proxy servers here
        ]
        
        # Job search parameters
        self.search_locations = [
            "New York, NY", "San Francisco, CA", "Seattle, WA",
            "Austin, TX", "Boston, MA", "Chicago, IL"
        ]
        
        self.job_titles = [
            "software engineer", "software developer", 
            "data scientist", "machine learning engineer",
            "data engineer", "full stack developer"
        ]

    def setup_logging(self):
        """Configure logging for the ETL process."""
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('job_scraper.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def setup_database(self):
        """Create SQLite database and tables if they don't exist."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Create jobs table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS jobs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            job_id TEXT UNIQUE,
            title TEXT,
            company TEXT,
            company_url TEXT,
            job_url TEXT,
            location_country TEXT,
            location_city TEXT,
            location_state TEXT,
            description TEXT,
            job_type TEXT,
            salary_interval TEXT,
            salary_min_amount REAL,
            salary_max_amount REAL,
            salary_currency TEXT,
            date_posted TIMESTAMP,
            is_remote BOOLEAN,
            job_function TEXT,
            company_industry TEXT,
            source_site TEXT,
            scrape_date TIMESTAMP,
            UNIQUE(job_url, company, title)
        )
        """)
        
        conn.commit()
        conn.close()

    def generate_job_id(self, row: Dict[str, Any]) -> str:
        """Generate a unique job ID based on job details."""
        components = [
            str(row.get('job_url', '')),
            str(row.get('company', '')),
            str(row.get('title', '')),
            str(row.get('location_city', '')),
            str(row.get('date_posted', ''))
        ]
        return '_'.join(components)

    def scrape_jobs_for_location(self, location: str, search_term: str) -> pd.DataFrame:
        """Scrape jobs for a specific location and search term."""
        try:
            self.logger.info(f"Scraping jobs for {search_term} in {location}")
            
            jobs = scrape_jobs(
                site_name=["indeed", "linkedin", "zip_recruiter", "glassdoor"],
                search_term=search_term,
                location=location,
                results_wanted=1000,  # Adjust based on your needs
                hours_old=72,
                country_indeed='USA',
                proxies=self.proxies,
                verbose=1
            )
            
            if jobs is not None and not jobs.empty:
                jobs['scrape_date'] = datetime.now()
                jobs['source_site'] = jobs['site']
                return jobs
            
            return pd.DataFrame()
            
        except Exception as e:
            self.logger.error(f"Error scraping jobs for {location}: {str(e)}")
            return pd.DataFrame()

    def transform_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transform and clean the scraped data."""
        if df.empty:
            return df
            
        # Generate unique job IDs
        df['job_id'] = df.apply(self.generate_job_id, axis=1)
        
        # Clean salary data
        df['salary_min_amount'] = pd.to_numeric(df['min_amount'], errors='coerce')
        df['salary_max_amount'] = pd.to_numeric(df['max_amount'], errors='coerce')
        df['salary_interval'] = df['interval']
        
        # Clean and standardize columns
        df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')
        df['scrape_date'] = pd.to_datetime(df['scrape_date'])
        
        # Select and rename columns for database
        columns_mapping = {
            'job_id': 'job_id',
            'title': 'title',
            'company': 'company',
            'company_url': 'company_url',
            'job_url': 'job_url',
            'country': 'location_country',
            'city': 'location_city',
            'state': 'location_state',
            'description': 'description',
            'job_type': 'job_type',
            'salary_interval': 'salary_interval',
            'salary_min_amount': 'salary_min_amount',
            'salary_max_amount': 'salary_max_amount',
            'currency': 'salary_currency',
            'date_posted': 'date_posted',
            'is_remote': 'is_remote',
            'job_function': 'job_function',
            'company_industry': 'company_industry',
            'source_site': 'source_site',
            'scrape_date': 'scrape_date'
        }
        
        return df.rename(columns=columns_mapping)[list(columns_mapping.values())]

    def load_to_database(self, df: pd.DataFrame):
        """Load transformed data into SQLite database."""
        if df.empty:
            self.logger.warning("No data to load into database")
            return
            
        try:
            conn = sqlite3.connect(self.db_path)
            
            # Insert new records, ignore duplicates
            df.to_sql('jobs', conn, if_exists='append', index=False, 
                     method='multi', chunksize=1000)
            
            conn.commit()
            self.logger.info(f"Successfully loaded {len(df)} jobs into database")
            
        except Exception as e:
            self.logger.error(f"Error loading data to database: {str(e)}")
            
        finally:
            conn.close()

    def run_etl_pipeline(self):
        """Execute the complete ETL pipeline."""
        total_jobs = 0
        
        for location in self.search_locations:
            for job_title in self.job_titles:
                try:
                    # Add random delay between searches to avoid rate limiting
                    time.sleep(random.uniform(2, 5))
                    
                    # Extract
                    raw_data = self.scrape_jobs_for_location(location, job_title)
                    
                    if not raw_data.empty:
                        # Transform
                        transformed_data = self.transform_data(raw_data)
                        
                        # Load
                        self.load_to_database(transformed_data)
                        
                        total_jobs += len(transformed_data)
                        
                except Exception as e:
                    self.logger.error(f"Pipeline error for {job_title} in {location}: {str(e)}")
                    continue
        
        self.logger.info(f"ETL pipeline completed. Total jobs processed: {total_jobs}")

if __name__ == "__main__":
    # Initialize and run the ETL pipeline
    etl = JobScraperETL()
    etl.run_etl_pipeline()

2025-01-31 23:27:22,593 - INFO - Scraping jobs for software engineer in New York, NY
